In [70]:
import pandas as pd
import numpy as np

FEATURE_PATH = "../data/processed/"
DASHBOARD_PATH = "../data/dashboard/"
OUTPUT_PATH = "../data/dashboard/"
MODEL_PATH = "../models/"
features = pd.read_csv(
    FEATURE_PATH + "features_ca1.csv",
    parse_dates=["date"]
)

product_stats = pd.read_csv(
    FEATURE_PATH + "product_stats_ca1.csv"
)

C:\Users\HP\AppData\Local\Temp\ipykernel_10424\793047828.py:8: DtypeWarning: Columns (15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  features = pd.read_csv(


In [ ]:
import joblib

from sklearn.metrics import mean_absolute_error, mean_squared_error
from prophet import Prophet

In [ ]:
features.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,event_type_1,event_name_2,event_type_2,snap_CA,lag_1,lag_7,lag_14,rolling_mean_7,rolling_mean_14,is_weekend
0,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,1
1,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_2,0,2011-01-30,11101,...,NaN,NaN,NaN,0,3.0,NaN,NaN,NaN,NaN,0
2,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_3,0,2011-01-31,11101,...,NaN,NaN,NaN,0,0.0,NaN,NaN,NaN,NaN,0
3,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_4,1,2011-02-01,11101,...,NaN,NaN,NaN,1,0.0,NaN,NaN,NaN,NaN,0
4,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_5,4,2011-02-02,11101,...,NaN,NaN,NaN,1,1.0,NaN,NaN,NaN,NaN,0


In [ ]:
product_stats.head()

,item_id,avg_sales,std_sales,zero_ratio,segment
0,FOODS_1_001,0.783316,1.256481,0.564008,intermittent
1,FOODS_1_002,0.477440,0.804688,0.668940,intermittent
2,FOODS_1_003,0.832634,1.163669,0.528332,intermittent
3,FOODS_1_004,8.288038,8.868324,0.319517,medium
4,FOODS_1_005,1.155824,2.009782,0.539874,intermittent


In [ ]:
df = features.merge(
    product_stats,
    on="item_id",
    how="left"
)

In [ ]:
df.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,lag_1,lag_7,lag_14,rolling_mean_7,rolling_mean_14,is_weekend,avg_sales,std_sales,zero_ratio,segment
0,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,...,NaN,NaN,NaN,NaN,NaN,1,0.783316,1.256481,0.564008,intermittent
1,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_2,0,2011-01-30,11101,...,3.0,NaN,NaN,NaN,NaN,0,0.783316,1.256481,0.564008,intermittent
2,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_3,0,2011-01-31,11101,...,0.0,NaN,NaN,NaN,NaN,0,0.783316,1.256481,0.564008,intermittent
3,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_4,1,2011-02-01,11101,...,0.0,NaN,NaN,NaN,NaN,0,0.783316,1.256481,0.564008,intermittent
4,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_5,4,2011-02-02,11101,...,1.0,NaN,NaN,NaN,NaN,0,0.783316,1.256481,0.564008,intermittent


In [ ]:
df["segment"].value_counts()

segment
intermittent          4241121
medium                1096149
smooth_high_volume     495467
Name: count, dtype: int64

In [ ]:
split_date = df["date"].quantile(0.8)

test_df = df[df["date"] > split_date].copy()

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    bias = np.mean(y_pred - y_true)
    return mae, rmse, bias

In [ ]:
rolling_df = test_df[test_df["segment"] == "intermittent"].copy()

rolling_df["prediction"] = rolling_df["rolling_mean_7"]
rolling_df["model_used"] = "Rolling7"

rolling_preds = rolling_df[
    ["item_id", "date", "sales", "prediction", "model_used"]
]

In [ ]:
smooth_items = product_stats[
    product_stats["segment"] == "smooth_high_volume"
]["item_id"]

prophet_outputs = []

for item in smooth_items:
    train_item = df[
        (df["item_id"] == item) &
        (df["date"] <= split_date)
    ][["date", "sales"]]

    test_item = df[
        (df["item_id"] == item) &
        (df["date"] > split_date)
    ][["date", "sales"]]

    if len(test_item) == 0:
        continue

    prophet_train = train_item.rename(columns={"date": "ds", "sales": "y"})
    prophet_future = test_item[["date"]].rename(columns={"date": "ds"})

    prophet_model = Prophet(
        weekly_seasonality=True,
        yearly_seasonality=True,
        daily_seasonality=False
    )

    prophet_model.fit(prophet_train)
    forecast = prophet_model.predict(prophet_future)

    tmp = test_item.copy()
    tmp["prediction"] = forecast["yhat"].values
    tmp["model_used"] = "Prophet"

    prophet_outputs.append(tmp)

17:52:10 - cmdstanpy - INFO - Chain [1] start processing
17:52:10 - cmdstanpy - INFO - Chain [1] done processing
17:52:12 - cmdstanpy - INFO - Chain [1] start processing
17:52:12 - cmdstanpy - INFO - Chain [1] done processing
17:52:14 - cmdstanpy - INFO - Chain [1] start processing
17:52:14 - cmdstanpy - INFO - Chain [1] done processing
17:52:16 - cmdstanpy - INFO - Chain [1] start processing
17:52:17 - cmdstanpy - INFO - Chain [1] done processing
17:52:18 - cmdstanpy - INFO - Chain [1] start processing
17:52:18 - cmdstanpy - INFO - Chain [1] done processing
17:52:20 - cmdstanpy - INFO - Chain [1] start processing
17:52:20 - cmdstanpy - INFO - Chain [1] done processing
17:52:22 - cmdstanpy - INFO - Chain [1] start processing
17:52:22 - cmdstanpy - INFO - Chain [1] done processing
17:52:24 - cmdstanpy - INFO - Chain [1] start processing
17:52:24 - cmdstanpy - INFO - Chain [1] done processing
17:52:26 - cmdstanpy - INFO - Chain [1] start processing
17:52:26 - cmdstanpy - INFO - Chain [1]

In [ ]:
prophet_preds = pd.concat(prophet_outputs, ignore_index=True)


In [47]:
lgb_model = joblib.load(
    MODEL_PATH + "lightgbm_medium_ca1.pkl"
)

In [48]:
type(lgb_model)

lightgbm.sklearn.LGBMRegressor

In [53]:
prices = pd.read_csv("../data/raw/sell_prices.csv")

df = df.merge(
    prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

In [54]:
df = df.sort_values(["item_id", "date"])

df["price_change"] = (
    df.groupby("item_id")["sell_price"]
    .pct_change()
    .fillna(0)
)

C:\Users\HP\AppData\Local\Temp\ipykernel_10424\3431983201.py:5: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()


In [55]:
item_stats = (
    df.groupby("item_id")
    .agg(
        item_mean_sales=("sales", "mean"),
        item_zero_ratio=("sales", lambda x: (x == 0).mean())
    )
    .reset_index()
)

df = df.merge(item_stats, on="item_id", how="left")

In [56]:
df["sales_norm"] = (
    df["sales"] / df["item_mean_sales"]
)

In [57]:
feature_cols = [
    "lag_1", "lag_7", "lag_14",
    "rolling_mean_7", "rolling_mean_14",
    "sell_price", "price_change",
    "wday", "month",
    "item_mean_sales", "item_zero_ratio"
]

target_col = "sales_norm"

In [58]:
X_test = df[feature_cols]

In [59]:
y_pred = lgb_model.predict(X_test)

In [60]:
df["prediction"] = y_pred
df["model_used"] = "LightGBM"

In [62]:
lgb_preds = df[
    ["item_id", "date", "sales", "prediction", "model_used"]
].copy()

In [63]:
all_predictions = pd.concat(
    [rolling_preds, prophet_preds, lgb_preds],
    ignore_index=True
)

In [64]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    bias = np.mean(y_pred - y_true)
    return mae, rmse, bias

In [65]:
segment_metrics = (
    all_predictions
    .merge(product_stats, on="item_id", how="left")
    .groupby(["segment", "model_used"])
    .apply(lambda x: pd.Series(
        evaluate(x["sales"], x["prediction"]),
        index=["mae", "rmse", "bias"]
    ))
    .reset_index()
)

C:\Users\HP\AppData\Local\Temp\ipykernel_10424\2215898783.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(


In [67]:
product_metrics = (
    all_predictions
    .groupby(["item_id", "model_used"])
    .apply(lambda x: pd.Series(
        evaluate(x["sales"], x["prediction"]),
        index=["mae", "rmse", "bias"]
    ))
    .reset_index()
    .merge(product_stats, on="item_id", how="left")
)

C:\Users\HP\AppData\Local\Temp\ipykernel_10424\3231756038.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(


In [68]:
all_predictions["abs_error"] = (
    all_predictions["prediction"] - all_predictions["sales"]
).abs()

reliability_metrics = (
    all_predictions
    .groupby("model_used")
    .agg(
        pct_within_1=("abs_error", lambda x: (x <= 1).mean())
    )
    .reset_index()
)

In [71]:
all_predictions.to_csv(
    OUTPUT_PATH + "forecast_vs_actual.csv",
    index=False
)

segment_metrics.to_csv(
    OUTPUT_PATH + "segment_metrics.csv",
    index=False
)

product_metrics.to_csv(
    OUTPUT_PATH + "product_metrics.csv",
    index=False
)

reliability_metrics.to_csv(
    OUTPUT_PATH + "reliability_metrics.csv",
    index=False
)